# Projeto Aplicado III – Sistema de Recomendação de Filmes

## Etapa 4 – Resultados, Análise Crítica, Conclusão e Trabalhos Futuros

**Integrantes (Grupo Projeto 17):**
- Daniel Silva Souza Filho (RA: 10737593)
- Diogo Ferreira da Silva (RA: 10737944)

Este notebook contempla o pipeline completo do projeto (Análise Exploratória, Preparação dos Dados, Baseline de Popularidade, Filtragem Colaborativa Inicial e Filtragem Colaborativa Ajustada por Regularização), acrescido das **visualizações de desempenho geral do modelo**, **análise comparativa** e **módulo de demonstração para aplicação na comunidade**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity

import urllib.request
import zipfile

# Download da base MovieLens
url = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
urllib.request.urlretrieve(url, "movielens.zip")

with zipfile.ZipFile("movielens.zip", "r") as arquivo:
    arquivo.extractall(".")

# Carregamento dos dados
ratings = pd.read_csv("ml-latest-small/ratings.csv")
movies = pd.read_csv("ml-latest-small/movies.csv")

print("Base carregada com sucesso!")


## 1. Análise Exploratória

Inicialmente, será realizada uma análise da base para verificar sua dimensão,
a distribuição das avaliações e a existência de valores ausentes ou registros
duplicados.


In [ ]:
print("Avaliações:", len(ratings))
print("Usuários:", ratings["userId"].nunique())
print("Filmes avaliados:", ratings["movieId"].nunique())
print("Filmes cadastrados:", movies["movieId"].nunique())

print("\nValores ausentes:")
print(ratings.isnull().sum())

print("\nDuplicados:", ratings.duplicated().sum())

In [ ]:
ratings["rating"].value_counts().sort_index().plot(kind="bar")

plt.title("Distribuição das avaliações")
plt.xlabel("Nota")
plt.ylabel("Quantidade")
plt.show()

## 2. Preparação dos Dados

Como não foram encontrados valores ausentes nem registros duplicados, não foi
necessário remover observações. Os dados serão separados em 80% para treinamento
e 20% para teste.

Para a avaliação das recomendações, serão considerados relevantes os filmes
que receberam nota igual ou superior a 4,0.


In [ ]:
train, test = train_test_split(
    ratings,
    test_size=0.2,
    random_state=42
)

LIMIAR = 4.0

print("Treino:", len(train))
print("Teste:", len(test))


## 3. Baseline de Popularidade

Como referência, será utilizado um modelo simples que recomenda os filmes
mais populares do conjunto de treinamento. Para cada usuário, são excluídos
os filmes que ele já avaliou.


In [ ]:
# Filmes ordenados pela quantidade de avaliações
populares = train["movieId"].value_counts()

def recomendar_populares(user_id, k=10):
    vistos = set(
        train[train["userId"] == user_id]["movieId"]
    )

    return [
        filme for filme in populares.index
        if filme not in vistos
    ][:k]

In [ ]:
rec_pop = recomendar_populares(1)

movies[
    movies["movieId"].isin(rec_pop)
][["movieId", "title"]]


## 4. Filtragem Colaborativa

O modelo principal utiliza filtragem colaborativa baseada em usuários.
Inicialmente, é construída uma matriz usuário-item. Em seguida, a similaridade
entre os usuários é calculada por meio da similaridade do cosseno.

As recomendações são obtidas a partir das avaliações realizadas pelos usuários
mais semelhantes.


In [ ]:
matriz = train.pivot_table(
    index="userId",
    columns="movieId",
    values="rating"
)

similaridade = pd.DataFrame(
    cosine_similarity(matriz.fillna(0)),
    index=matriz.index,
    columns=matriz.index
)

print("Dimensão da matriz:", matriz.shape)


In [ ]:
def recomendar_colaborativo(user_id, k=10):
    # 50 usuários mais semelhantes
    vizinhos = (
        similaridade[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
        .head(50)
    )

    vistos = set(matriz.loc[user_id].dropna().index)

    scores = {}

    for vizinho, sim in vizinhos.items():
        avaliacoes = matriz.loc[vizinho].dropna()

        for filme, nota in avaliacoes.items():
            if filme not in vistos:
                scores.setdefault(filme, [])
                scores[filme].append((sim, nota))

    # Mantém filmes avaliados por pelo menos 3 vizinhos
    resultado = {}

    for filme, valores in scores.items():
        if len(valores) >= 3:
            soma = sum(sim * nota for sim, nota in valores)
            pesos = sum(sim for sim, nota in valores)

            if pesos > 0:
                resultado[filme] = soma / pesos

    return [
        filme for filme, score in sorted(
            resultado.items(),
            key=lambda x: x[1],
            reverse=True
        )[:k]
    ]


In [ ]:
rec_cf = recomendar_colaborativo(1)

movies[
    movies["movieId"].isin(rec_cf)
][["movieId", "title"]]


## 5. Avaliação

Os modelos serão comparados utilizando Precision@10, Recall@10 e NDCG@10,
métricas adequadas para avaliar listas ordenadas de recomendações.


In [ ]:
def metricas(recomendados, relevantes, k=10):
    recomendados = recomendados[:k]
    relevantes = set(relevantes)

    acertos = set(recomendados) & relevantes

    precision = len(acertos) / k
    recall = len(acertos) / len(relevantes) if relevantes else 0

    dcg = sum(
        1 / np.log2(i + 2)
        for i, filme in enumerate(recomendados)
        if filme in relevantes
    )

    ideal = min(len(relevantes), k)
    idcg = sum(
        1 / np.log2(i + 2)
        for i in range(ideal)
    )

    ndcg = dcg / idcg if idcg else 0

    return precision, recall, ndcg


In [ ]:
resultados = []

for usuario in test["userId"].unique():

    relevantes = test[
        (test["userId"] == usuario) &
        (test["rating"] >= LIMIAR)
    ]["movieId"].tolist()

    if not relevantes:
        continue

    pop = recomendar_populares(usuario)
    cf = recomendar_colaborativo(usuario)

    resultados.append(
        metricas(pop, relevantes) +
        metricas(cf, relevantes)
    )

resultado = pd.DataFrame(
    resultados,
    columns=[
        "P_Pop", "R_Pop", "NDCG_Pop",
        "P_CF", "R_CF", "NDCG_CF"
    ]
)

comparacao = pd.DataFrame({
    "Modelo": ["Popularidade", "Filtragem Colaborativa"],
    "Precision@10": [
        resultado["P_Pop"].mean(),
        resultado["P_CF"].mean()
    ],
    "Recall@10": [
        resultado["R_Pop"].mean(),
        resultado["R_CF"].mean()
    ],
    "NDCG@10": [
        resultado["NDCG_Pop"].mean(),
        resultado["NDCG_CF"].mean()
    ]
})

comparacao.round(4)


## 6. Resultados Preliminares

A prova de conceito permitiu comparar o modelo de filtragem colaborativa com
um baseline baseado na popularidade dos filmes.

O modelo de popularidade apresentou Precision@10 de 0,1215, Recall@10 de
0,1026 e NDCG@10 de 0,1579. Já a filtragem colaborativa apresentou
Precision@10 de 0,0230, Recall@10 de 0,0308 e NDCG@10 de 0,0273.

Nesta configuração inicial, o modelo baseado em popularidade apresentou
melhor desempenho nas três métricas avaliadas. Esse resultado indica que a
filtragem colaborativa ainda pode ser aprimorada por meio de ajustes nos
parâmetros e na estratégia de recomendação.

Como esta etapa corresponde a uma prova de conceito, os resultados obtidos
servem como referência para os refinamentos que poderão ser realizados nas
próximas etapas do projeto.


## 7. Refinamento do Modelo

Na Etapa 2, a filtragem colaborativa apresentou desempenho inferior ao
baseline de popularidade.

Nesta etapa, será testado um ajuste na pontuação das recomendações,
considerando a quantidade de avaliações realizadas pelos usuários
semelhantes.

O objetivo é reduzir a influência de filmes avaliados por poucos
vizinhos e verificar se o ajuste melhora o desempenho do modelo.


In [ ]:
def recomendar_ajustado(user_id, k=10):
    vizinhos = (
        similaridade[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
        .head(50)
    )

    vistos = set(matriz.loc[user_id].dropna().index)
    scores = {}

    for vizinho, sim in vizinhos.items():
        for filme, nota in matriz.loc[vizinho].dropna().items():
            if filme not in vistos and sim > 0:
                scores.setdefault(filme, [])
                scores[filme].append((sim, nota))

    resultado = {}

    for filme, valores in scores.items():
        if len(valores) >= 3:
            soma = sum(sim * nota for sim, nota in valores)
            pesos = sum(sim for sim, nota in valores)

            previsao = soma / pesos

            # Regularização pela quantidade de avaliações
            n = len(valores)
            score = (n * previsao + 10 * 3.5) / (n + 10)

            resultado[filme] = score

    return [
        filme for filme, score in sorted(
            resultado.items(),
            key=lambda x: x[1],
            reverse=True
        )[:k]
    ]


In [ ]:
resultados_ajustados = []

for usuario in test["userId"].unique():

    relevantes = test[
        (test["userId"] == usuario) &
        (test["rating"] >= LIMIAR)
    ]["movieId"].tolist()

    if not relevantes:
        continue

    recomendados = recomendar_ajustado(usuario)

    resultados_ajustados.append(
        metricas(recomendados, relevantes)
    )

resultado_ajustado = pd.DataFrame(
    resultados_ajustados,
    columns=["Precision@10", "Recall@10", "NDCG@10"]
)

resultado_ajustado.mean().round(4)


In [ ]:
comparacao_etapa3 = pd.concat([
    comparacao,
    pd.DataFrame([{
        "Modelo": "Filtragem Colaborativa Ajustada",
        "Precision@10": resultado_ajustado["Precision@10"].mean(),
        "Recall@10": resultado_ajustado["Recall@10"].mean(),
        "NDCG@10": resultado_ajustado["NDCG@10"].mean()
    }])
], ignore_index=True)

display(comparacao_etapa3.round(4))


## 8. Visualização de Dados do Desempenho dos Modelos (Etapa 4)

Para representar visualmente o desempenho geral dos modelos avaliados e o impacto da regularização sobre as métricas Top-10 (`Precision@10`, `Recall@10` e `NDCG@10`), são gerados a seguir o gráfico comparativo de barras agrupadas e o painel de análise do amortecimento e ganho percentual sobre o baseline.


In [ ]:
# Gráfico comparativo de desempenho entre os três modelos (Precision@10, Recall@10 e NDCG@10)
metricas_nomes = ["Precision@10", "Recall@10", "NDCG@10"]
valores_pop = [comparacao_etapa3.loc[0, m] for m in metricas_nomes]
valores_cf = [comparacao_etapa3.loc[1, m] for m in metricas_nomes]
valores_adj = [comparacao_etapa3.loc[2, m] for m in metricas_nomes]

x = np.arange(len(metricas_nomes))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
rects1 = ax.bar(x - width, valores_pop, width, label="Baseline de Popularidade", color="#475569")
rects2 = ax.bar(x, valores_cf, width, label="Filtragem Colaborativa (Inicial)", color="#d97706")
rects3 = ax.bar(x + width, valores_adj, width, label="Filtragem Colaborativa Ajustada", color="#b5121b")

ax.set_ylabel("Valor Médio da Métrica")
ax.set_title("Desempenho Geral dos Modelos nas Métricas Top-10")
ax.set_xticks(x)
ax.set_xticklabels(metricas_nomes)
ax.set_ylim(0, 0.24)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.legend()

for group in [rects1, rects2, rects3]:
    for rect in group:
        h = rect.get_height()
        ax.annotate(f"{h:.4f}", xy=(rect.get_x() + rect.get_width()/2, h),
                    xytext=(0, 3), textcoords="offset points", ha="center", va="bottom")
plt.tight_layout()
plt.show()


In [ ]:
# Painel analítico: (a) Efeito do número de vizinhos n no Score regularizado e (b) Ganho percentual sobre o Baseline
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

n_vals = np.linspace(3, 50, 100)
for prev, cor, rotulo in [(5.0, "#b5121b", "Previsão = 5.0"), (4.5, "#2563eb", "Previsão = 4.5"), (4.0, "#059669", "Previsão = 4.0")]:
    score_vals = (n_vals * prev + 10 * 3.5) / (n_vals + 10)
    ax1.plot(n_vals, score_vals, label=rotulo, linewidth=2, color=cor)
ax1.axhline(3.5, color="gray", linestyle=":", label="Referência (3.5)")
ax1.set_title("(a) Amortecimento pela Regularização")
ax1.set_xlabel("Quantidade de vizinhos avaliadores (n)")
ax1.set_ylabel("Score Regularizado")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.legend()

ganhos_pct = [((a - b) / b) * 100 for a, b in zip(valores_adj, valores_pop)]
bars = ax2.bar(metricas_nomes, ganhos_pct, color=["#1e3a8a", "#0f766e", "#b5121b"], width=0.5)
ax2.set_title("(b) Ganho da FC Ajustada sobre o Baseline (%)")
ax2.set_ylabel("Melhoria Percentual (%)")
ax2.set_ylim(0, 45)
ax2.grid(axis="y", linestyle="--", alpha=0.4)
for bar in bars:
    h = bar.get_height()
    ax2.annotate(f"+{h:.2f}%", xy=(bar.get_x() + bar.get_width()/2, h),
                xytext=(0, 3), textcoords="offset points", ha="center", va="bottom")
plt.tight_layout()
plt.show()


## 9. Demonstração Prática: Aplicação do Projeto na Comunidade

A função abaixo permite aplicar o sistema de recomendação em tempo real com participantes da comunidade. A partir das notas atribuídas por um participante a filmes conhecidos do catálogo, o algoritmo calcula a similaridade do cosseno com os 610 usuários da base MovieLens e retorna as 10 recomendações personalizadas geradas pela **Filtragem Colaborativa Ajustada**.


In [ ]:
def recomendar_para_participante_comunidade(avaliacoes_participante, k=10):
    """
    Recebe um dicionário {movieId: nota} informado por um participante real da comunidade
    e retorna um DataFrame com os Top-K filmes recomendados pela Filtragem Colaborativa Ajustada.
    """
    vetor_usuario = pd.Series(0.0, index=matriz.columns)
    for m_id, nota in avaliacoes_participante.items():
        if m_id in vetor_usuario.index:
            vetor_usuario[m_id] = nota

    sims = cosine_similarity(vetor_usuario.values.reshape(1, -1), matriz.fillna(0).values)[0]
    vizinhos = pd.Series(sims, index=matriz.index).sort_values(ascending=False).head(50)

    vistos = set(avaliacoes_participante.keys())
    scores = {}
    for vizinho, sim in vizinhos.items():
        if sim <= 0:
            continue
        for filme, nota in matriz.loc[vizinho].dropna().items():
            if filme not in vistos:
                scores.setdefault(filme, []).append((sim, nota))

    resultado = []
    for filme, valores in scores.items():
        if len(valores) >= 3:
            soma = sum(s * n for s, n in valores)
            pesos = sum(s for s, _ in valores)
            previsao = soma / pesos
            n = len(valores)
            score_reg = (n * previsao + 10 * 3.5) / (n + 10)
            resultado.append((filme, score_reg, n, round(previsao, 2)))

    resultado.sort(key=lambda x: x[1], reverse=True)
    top_k = pd.DataFrame(resultado[:k], columns=["movieId", "Score_Regularizado", "Qtd_Vizinhos", "Media_Ponderada"])
    return top_k.merge(movies[["movieId", "title", "genres"]], on="movieId", how="left")

# Exemplo de uso na demonstração com a comunidade:
# Participante avaliou: Toy Story (1): 5.0, Matrix (2571): 5.0, Forrest Gump (356): 4.5, Shrek (4306): 4.5
perfil_exemplo = {1: 5.0, 2571: 5.0, 356: 4.5, 4306: 4.5}
recomendar_para_participante_comunidade(perfil_exemplo, k=10)
